# 🗄️ Database Setup for Rice Production Data
## SQLite Database Creation and Configuration

**Objective:** Create a structured SQLite database to store cleaned paddy production data, enabling efficient querying and data retrieval for forecasting models and dashboard applications.

**Why SQLite?**
- Lightweight and serverless
- No complex configuration required
- Perfect for local development and small to medium datasets
- Easy integration with Python and web frameworks

**Database Schema:** Single table `production_stats` containing all cleaned production records with district, season, and temporal information.

---
## 1. Import Required Libraries

In [4]:
import pandas as pd
import sqlite3
import os
from datetime import datetime

print("✅ Libraries imported successfully")
print(f"📅 Setup Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully
📅 Setup Date: 2026-01-26 13:23:14


---
## 2. Load Cleaned Data

Loading the preprocessed paddy production data from CSV file.

In [13]:
# Define file path
csv_path = os.path.join('..', 'data', 'processed', 'paddy_data_cleaned.csv')

# Check if file exists
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Data file not found at: {csv_path}")

# Load data
df = pd.read_csv(csv_path)

print("✅ Data loaded successfully")
print(f"📊 Total Records: {len(df):,}")
print(f"📅 Date Range: {df['Year'].min()} to {df['Year'].max()}")
print(f"🔢 Columns: {', '.join(df.columns)}")
print("\n" + "="*70)
display(df.head())

✅ Data loaded successfully
📊 Total Records: 1,000
📅 Date Range: 2004 - 2005 to 2023
🔢 Columns: District, Major_Schemes_Sown, Minor_Schemes_Sown, Rainfed_Sown, All_Schemes_Sown, Major_Schemes_Harvested, Minor_Schemes_Harvested, Rainfed_Harvested, All_Schemes_Harvested, Major_Schemes_Yield, Minor_Schemes_Yield, Rainfed_Yield, Average_Yield, Nett_Extent_Harvested, Total_Production, Source_File, Season, Year



,District,Major_Schemes_Sown,Minor_Schemes_Sown,Rainfed_Sown,All_Schemes_Sown,Major_Schemes_Harvested,Minor_Schemes_Harvested,Rainfed_Harvested,All_Schemes_Harvested,Major_Schemes_Yield,Minor_Schemes_Yield,Rainfed_Yield,Average_Yield,Nett_Extent_Harvested,Total_Production,Source_File,Season,Year
0,COLOMBO,1141.0,85.0,3367.0,4594.0,85.0,1141.0,3365.0,4592.0,2564.0,3426.0,3218.0,3251.0,3903.0,12687.0,2004 - 2005 Maha.csv,Maha,2004 - 2005
1,GAMPAHA,1170.0,1192.0,7811.0,10173.0,1191.0,1170.0,7809.0,10170.0,3226.0,3500.0,3430.0,3408.0,8644.0,29460.0,2004 - 2005 Maha.csv,Maha,2004 - 2005
2,KALUTARA,1938.0,96.0,11608.0,13642.0,93.0,1936.0,11581.0,13610.0,3344.0,2594.0,2913.0,2865.0,12556.0,35971.0,2004 - 2005 Maha.csv,Maha,2004 - 2005
3,GALLE,175.0,0.0,14323.0,14498.0,0.0,174.0,13769.0,13943.0,0.0,3614.0,3719.0,3711.0,10912.0,40494.0,2004 - 2005 Maha.csv,Maha,2004 - 2005
4,MATARA,2809.0,3615.0,8439.0,14862.0,3610.0,2752.0,8333.0,14695.0,4415.0,3859.0,3498.0,3785.0,11602.0,43912.0,2004 - 2005 Maha.csv,Maha,2004 - 2005


---
## 3. Create SQLite Database Connection

Establishing connection to SQLite database. If the database doesn't exist, SQLite will create it automatically.

In [14]:
# Define database path
db_path = os.path.join('..', 'data', 'rice_db.sqlite')

# Create data directory if it doesn't exist
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# Establish connection
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("🔌 Database connection established")


🔌 Database connection established


---
## 4. Import Data to Database

Creating the `production_stats` table and importing cleaned data. Using `if_exists='replace'` ensures the table is recreated if it already exists, preventing duplicate data.

In [7]:
# Import data to SQLite table
table_name = 'production_stats'

print(f"📥 Importing {len(df):,} records into '{table_name}' table...")

# Write DataFrame to SQL
df.to_sql(table_name, conn, if_exists='replace', index=False)

print("✅ Data successfully imported to database")

# Verify table creation
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"📋 Tables in database: {[table[0] for table in tables]}")

# Get row count
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
row_count = cursor.fetchone()[0]
print(f"📊 Total rows in '{table_name}': {row_count:,}")

📥 Importing 1,000 records into 'production_stats' table...
✅ Data successfully imported to database
📋 Tables in database: ['production_stats']
📊 Total rows in 'production_stats': 1,000


---
## 5. Verify Database Schema

Inspecting the database structure to confirm proper table creation and column definitions.

In [8]:
# Get table schema information
cursor.execute(f"PRAGMA table_info({table_name})")
schema = cursor.fetchall()

print("📋 Database Schema:")
print("="*70)
print(f"{'Column Name':<30} {'Type':<15} {'Nullable':<10}")
print("-"*70)

for col in schema:
    col_id, col_name, col_type, not_null, default_val, pk = col
    nullable = "No" if not_null else "Yes"
    print(f"{col_name:<30} {col_type:<15} {nullable:<10}")

print("="*70)

📋 Database Schema:
Column Name                    Type            Nullable  
----------------------------------------------------------------------
District                       TEXT            Yes       
Major_Schemes_Sown             REAL            Yes       
Minor_Schemes_Sown             REAL            Yes       
Rainfed_Sown                   REAL            Yes       
All_Schemes_Sown               REAL            Yes       
Major_Schemes_Harvested        REAL            Yes       
Minor_Schemes_Harvested        REAL            Yes       
Rainfed_Harvested              REAL            Yes       
All_Schemes_Harvested          REAL            Yes       
Major_Schemes_Yield            REAL            Yes       
Minor_Schemes_Yield            REAL            Yes       
Rainfed_Yield                  REAL            Yes       
Average_Yield                  REAL            Yes       
Nett_Extent_Harvested          REAL            Yes       
Total_Production               REAL     

---
## 6. Test Database Queries

Running sample SQL queries to verify data integrity and database functionality.

### Test Query 1: Top 5 Production Records for Polonnaruwa District

In [9]:
query1 = """
SELECT 
    District, 
    Year, 
    Season, 
    Total_Production,
    Average_Yield
FROM production_stats 
WHERE District = 'POLONNARUWA' 
ORDER BY Total_Production DESC 
LIMIT 5
"""

result1 = pd.read_sql(query1, conn)

print("🔍 Top 5 Seasons for POLONNARUWA District")
print("="*70)
display(result1)

🔍 Top 5 Seasons for POLONNARUWA District


,District,Year,Season,Total_Production,Average_Yield
0,POLONNARUWA,2014-2015,Maha,349625.0,5306.0
1,POLONNARUWA,2019-2020,Maha,318352.0,5404.0
2,POLONNARUWA,2020-2021,Maha,307190.0,5220.0
3,POLONNARUWA,2021,Yala,300317.0,5441.0
4,POLONNARUWA,2019,Yala,299468.0,5511.0


### Test Query 2: Total Production by Season

In [10]:
query2 = """
SELECT 
    Season,
    COUNT(*) as Record_Count,
    SUM(Total_Production) as Total_Production,
    AVG(Total_Production) as Avg_Production,
    AVG(Average_Yield) as Avg_Yield
FROM production_stats 
GROUP BY Season
ORDER BY Total_Production DESC
"""

result2 = pd.read_sql(query2, conn)

print("🔍 Production Comparison by Season (All Years)")
print("="*70)
display(result2)

🔍 Production Comparison by Season (All Years)


,Season,Record_Count,Total_Production,Avg_Production,Avg_Yield
0,Maha,499,46595293.0,93377.340681,4008.234469
1,Yala,501,28941759.0,57767.982036,3857.868263


### Test Query 3: Top 10 Districts by Total Production

In [11]:
query3 = """
SELECT 
    District,
    SUM(Total_Production) as Total_Production,
    AVG(Average_Yield) as Avg_Yield,
    COUNT(*) as Total_Records
FROM production_stats 
WHERE District NOT LIKE '%SRI%'
GROUP BY District
ORDER BY Total_Production DESC
LIMIT 10
"""

result3 = pd.read_sql(query3, conn)

print("🔍 Top 10 Rice Producing Districts (All-Time)")
print("="*70)
display(result3)

🔍 Top 10 Rice Producing Districts (All-Time)


,District,Total_Production,Avg_Yield,Total_Records
0,AMPARA,10467861.0,4653.621622,37
1,POLONNARUWA,9320738.0,4903.675676,37
2,KURUNEGALA,8435250.0,3813.052632,38
3,ANURADHAPURA,8153947.0,4528.868421,38
4,HAMBANTOTA,4769015.0,5298.210526,38
5,BATTICALOA,4082078.0,3496.236842,38
6,TRINCOMALEE,3538328.0,4374.263158,38
7,MONARAGALA,3061133.0,4142.131579,38
8,BADULLA,2556845.0,4416.342105,38
9,MAHAWELI_H,2213254.0,5476.000000,26


---
## 7. Close Database Connection

Properly closing the database connection to prevent file locks and ensure data integrity.

In [12]:
# Close the database connection
conn.close()

print("✅ Database connection closed successfully")
print("\n" + "="*70)
print("📁 Database File: rice_db.sqlite")
print("📊 Table: production_stats")
print("✔️  Database setup complete and ready for use!")
print("="*70)

✅ Database connection closed successfully

📁 Database File: rice_db.sqlite
📊 Table: production_stats
✔️  Database setup complete and ready for use!


---
## 📝 Summary

### Database Setup Complete ✅

**What was accomplished:**
1. ✔️ Created SQLite database (`rice_db.sqlite`)
2. ✔️ Imported cleaned paddy production data into `production_stats` table
3. ✔️ Verified database schema and structure
4. ✔️ Tested multiple SQL queries to ensure data integrity
5. ✔️ Confirmed proper connection management

### Database Details:
- **Location:** `data/rice_db.sqlite`
- **Table Name:** `production_stats`
- **Total Records:** 1,000+ entries
- **Columns:** Year, Season, District, Production metrics, Yield data

### Sample Queries Available:
- Filter by district and season
- Aggregate production statistics
- Time-series analysis
- District-level comparisons

